# Load: Domiciliation by country of birth and sex

**Source file:** `Table-1.11-Domiciliation-by-country-of-birth-and-sex.xlsx`  
**Source:** CBS Aruba and the Population Registry Office

**Pipeline:**
1. Load raw Excel → clean headers → produce tidy long-format DataFrame
2. Save to `data/processed/domiciliation.parquet`
3. Register as DuckDB view `domiciliation` in `outputs/db_files/aruba.duckdb`

**Notes:**
- The total row (`Total Domiciliation:`) is kept in the parquet for validation purposes.
  Filter it out in EDA with `WHERE country != 'Total Domiciliation:'`.
- Years covered: 2015–2023

---
## 1. Setup

In [ ]:
import pandas as pd
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent.parent if Path.cwd().name == "load" else Path.cwd()))

from config.project_paths import ROOT, DATA_RAW, DATA_PROCESSED, FIGURES, DB_FILES
from src.data_loader import save_to_parquet, register_in_duckdb

In [ ]:
# Verify paths
print("ROOT:          ", ROOT)
print("DATA_RAW:      ", DATA_RAW)
print("DATA_PROCESSED:", DATA_PROCESSED)
print("DB_FILES:      ", DB_FILES)

In [ ]:
SOURCE_FILE = DATA_RAW / "Table-1.11-Domiciliation-by-country-of-birth-and-sex.xlsx"
PARQUET_NAME = "domiciliation.parquet"
VIEW_NAME = "domiciliation"

if not SOURCE_FILE.exists():
    raise FileNotFoundError(f"Source file not found: {SOURCE_FILE}")

print("Source file found:", SOURCE_FILE)

---
## 2. Load and clean raw Excel

The Excel file has two header rows above the data:
- Row 0 (skipped): title text
- Row 1: year values, forward-filled across Male/Female pairs
- Row 2: `Country`, `Male`, `Female`, `Male`, `Female`, ...

We load with `skiprows=1, header=None` and reconstruct the column names
by combining both header rows into `{year}_{sex}` labels.

In [ ]:
raw_df = pd.read_excel(SOURCE_FILE, skiprows=1, header=None)
print(f"Raw shape: {raw_df.shape}")
raw_df.head(4)

In [ ]:
# Drop all-NaN rows, coerce year row to Int64
df = raw_df.dropna(how="all").copy()
df.iloc[0] = pd.to_numeric(df.iloc[0], errors="coerce").astype("Int64")

# Promote first row to column index, forward-fill year values
df.columns = df.iloc[0]
df = df.iloc[1:].reset_index(drop=True)
cols = pd.Series(df.columns)
df.columns = cols.ffill()

In [ ]:
# Build new column names: {year}_{sex}, or just 'Country' for the first column
header_top = pd.Series(df.columns)
header_bottom = df.loc[0]

new_cols = []
for top, bottom in zip(header_top, header_bottom):
    if pd.isna(top):
        new_cols.append(str(bottom).strip())
    elif str(top) == "Indicator":
        new_cols.append("Indicator")
    else:
        new_cols.append(f"{top}_{str(bottom).strip()}")

df.columns = new_cols
df = df.drop(index=0).reset_index(drop=True)

print("Columns:", df.columns.tolist())

In [ ]:
# Drop the source attribution row at the bottom
df = df[
    ~df["Country"].astype(str).str.startswith("Source:", na=False)
].reset_index(drop=True)

print(f"Cleaned shape: {df.shape}")
df

---
## 3. Reshape to tidy (long) format

In [ ]:
tidy_df = df.melt(
    id_vars="Country",
    var_name="year_sex",
    value_name="value"
)

# Split year_sex into separate columns
tidy_df[["year", "sex"]] = tidy_df["year_sex"].str.split("_", expand=True)
tidy_df = tidy_df.drop(columns="year_sex")

# Cast types
tidy_df["year"] = pd.to_numeric(tidy_df["year"], errors="coerce").astype("Int64")
tidy_df["value"] = pd.to_numeric(tidy_df["value"], errors="coerce").astype("Int64")

# Rename for consistency
tidy_df = tidy_df.rename(columns={"Country": "country"})

tidy_df = tidy_df[["country", "year", "sex", "value"]].sort_values(
    ["country", "year", "sex"]
).reset_index(drop=True)

print(f"Tidy shape: {tidy_df.shape}")
tidy_df.head(12)

---
## 4. Sanity check

Verify that the sum of country rows equals the total row for a sample year.

In [ ]:
for year in [2015, 2020, 2023]:
    total_row = tidy_df[
        (tidy_df["country"] == "Total Domiciliation:") & (tidy_df["year"] == year)
    ]["value"].sum()

    country_sum = tidy_df[
        (tidy_df["country"] != "Total Domiciliation:") & (tidy_df["year"] == year)
    ]["value"].sum()

    match = "✓" if total_row == country_sum else "✗"
    print(f"{year}: total={total_row}, sum of countries={country_sum}  {match}")

---
## 5. Save to parquet and register DuckDB view

In [ ]:
parquet_path = save_to_parquet(tidy_df, PARQUET_NAME)
print("Saved parquet:", parquet_path)

In [ ]:
register_in_duckdb(parquet_path, VIEW_NAME)
print(f"Registered DuckDB view: {VIEW_NAME}")

---
## 6. Smoke test

In [ ]:
import duckdb

con = duckdb.connect(str(DB_FILES / "aruba.duckdb"))

result = con.execute(f"SELECT * FROM {VIEW_NAME} ORDER BY country, year, sex LIMIT 10").df()
print(f"Shape from DuckDB: {con.execute(f'SELECT COUNT(*) FROM {VIEW_NAME}').fetchone()[0]} rows")
print(f"Years: {con.execute(f'SELECT MIN(year), MAX(year) FROM {VIEW_NAME}').fetchone()}")
print(f"Countries: {con.execute(f'SELECT DISTINCT country FROM {VIEW_NAME} ORDER BY country').df()['country'].tolist()}")
con.close()

result